In [ ]:
from datasets import load_dataset
from typing import TypedDict
from transformers import (
    GPT2Tokenizer, GPT2Config, GPT2LMHeadModel, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, pipeline
)

# Conjunto de Datos

In [ ]:
ds = load_dataset("bertin-project/alpaca-spanish")
ds

README.md:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

data/train-00000-of-00001-3a8ceb2c27df89(…): reconstructing file:   0%|          |  0.00B / 13.2MB            

data/train-00000-of-00001-3a8ceb2c27df89(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51942 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 51942
    })
})

In [ ]:
ds_full = ds["train"]
ds_full

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 51942
})

In [ ]:
ds_full[:3]

{'instruction': ['¿Qué significa DNA?',
  '¿Cuál es la capital de Francia?',
  'Identifica el que no encaja.'],
 'input': ['', '', 'Twitter, Instagram, Telegram'],
 'output': ['DNA significa ácido desoxirribonucleico.',
  'La capital de Francia es París.',
  'Telegram']}

In [ ]:
class DatasetSample(TypedDict):
    instruction: str
    input: str
    output: str

class TextSample(TypedDict):
    text: str

class TokenSample(TypedDict):
    input_ids: list[int]

In [ ]:
def template(sample: DatasetSample) -> TextSample:
    if sample["input"]:
        text = f"""<Instruction>
{sample["instruction"]}
</Instruction>
<Input>
{sample["input"]}
</Input>
<Output>
{sample["output"]}
</Output>
"""
    else:
        text = f"""<Instruction>
{sample["instruction"]}
</Instruction>
<Output>
{sample["output"]}
</Output>
"""
    return {"text": text}

In [ ]:
print(template(ds_full[0]))

{'text': '<Instruction>\n¿Qué significa DNA?\n</Instruction>\n<Output>\nDNA significa ácido desoxirribonucleico.\n</Output>\n'}


In [ ]:
ds_template = ds_full.map(
    template, num_proc=4, remove_columns=ds_full.column_names
)

Map (num_proc=4):   0%|          | 0/51942 [00:00<?, ? examples/s]

In [ ]:
ds_template[0]

{'text': '<Instruction>\n¿Qué significa DNA?\n</Instruction>\n<Output>\nDNA significa ácido desoxirribonucleico.\n</Output>\n'}

In [ ]:
ds_template

Dataset({
    features: ['text'],
    num_rows: 51942
})

# Validacion Cruzada

In [ ]:
ds_partition = ds_template.train_test_split(test_size=0.3, seed=0)
ds_partition

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 36359
    })
    test: Dataset({
        features: ['text'],
        num_rows: 15583
    })
})

# Tokenizador

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
tokenizer(ds_template[0]["text"], truncation=True, max_length=128)

{'input_ids': [27, 6310, 2762, 29, 198, 126, 123, 4507, 2634, 2216, 64, 7446, 30, 198, 3556, 6310, 2762, 29, 198, 27, 26410, 29, 198, 28886, 2216, 64, 6184, 94, 66, 17305, 748, 1140, 343, 822, 261, 14913, 3713, 13, 198, 3556, 26410, 29, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
def tokenize(sample: TextSample) -> TokenSample:
    tokens = tokenizer(sample["text"], truncation=True, max_length=128)
    tokens.pop("attention_mask")
    return tokens

ds_token = (
    ds_partition
    .map(tokenize, remove_columns=ds_template.column_names, num_proc=4)
    .with_format("torch")
)

Map (num_proc=4):   0%|          | 0/36359 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/15583 [00:00<?, ? examples/s]

In [ ]:
ds_token["train"][:2]

{'input_ids': tensor([[   27,  6310,  2762,    29,   198,  5842, 25440,  1288,  5128,   390,
            555,  1242,  8836,  3129,    78,   523,  4679,  1288, 26920,   615,
          19397,    11,  1152,    64,   555,   256,  8836,    83, 43348,  1827,
          23593,   823, 49443,  4533,   369,  8591, 19798, 22859,    13,   198,
           3556,  6310,  2762,    29,   198,    27, 20560,    29,   198, 14772,
            299,   518,  6862, 19798, 22859,   390,  7375, 11008,    12,  1129,
            387,   987,    81,   931, 17305,  8591,   410,  3755,   390,   881,
            292,  1048,   292,   551,   284,  4598,  1288, 27943,    78,    13,
           5401, 48484,   959, 39369,   289,   272,  1556,  4533,  6824, 25440,
            582,    68,  9491,  8591,  4902,   369,   269,   959,   411,  2276,
            528, 22484,   331,  1334,  1173,    66,   295,   274,   390,  2884,
          18015,    13,   198,  3556, 20560,    29,   198,    27, 26410,    29,
            198,     1,    

# Modelo

In [ ]:
config = GPT2Config.from_pretrained("openai-community/gpt2", n_ctx=128)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
model = GPT2LMHeadModel(config).to("cuda")

In [ ]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
n_params

124439808

# Entrenamiento

In [ ]:
tokenizer.eos_token

'<|endoftext|>'

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
sample = collator(ds_token["train"][:3]["input_ids"])
sample

{'input_ids': tensor([[   27,  6310,  2762,    29,   198,  5842, 25440,  1288,  5128,   390,
            555,  1242,  8836,  3129,    78,   523,  4679,  1288, 26920,   615,
          19397,    11,  1152,    64,   555,   256,  8836,    83, 43348,  1827,
          23593,   823, 49443,  4533,   369,  8591, 19798, 22859,    13,   198,
           3556,  6310,  2762,    29,   198,    27, 20560,    29,   198, 14772,
            299,   518,  6862, 19798, 22859,   390,  7375, 11008,    12,  1129,
            387,   987,    81,   931, 17305,  8591,   410,  3755,   390,   881,
            292,  1048,   292,   551,   284,  4598,  1288, 27943,    78,    13,
           5401, 48484,   959, 39369,   289,   272,  1556,  4533,  6824, 25440,
            582,    68,  9491,  8591,  4902,   369,   269,   959,   411,  2276,
            528, 22484,   331,  1334,  1173,    66,   295,   274,   390,  2884,
          18015,    13,   198,  3556, 20560,    29,   198,    27, 26410,    29,
            198,     1,    

In [ ]:
original = tokenizer.batch_decode(sample["input_ids"], skip_special_tokens=True)
original

['<Instruction>\nUsando el input de un artículo sobre el coronavirus, genera un título creativo relacionado con la pandemia.\n</Instruction>\n<Input>\nLa nueva pandemia de COVID-19 ha interrumpido la vida de muchas personas en todo el mundo. Los gobiernos han estado intentando manejar la crisis con cierres generalizados y restricciones de viaje.\n</Input>\n<Output>\n"Aguas desconoc',
 '<Instruction>\nAnalizar un ejemplo de conflicto en el texto dado.\n</Instruction>\n<Input>\nEn la película City Lights, una mujer adinerada ayuda a un hombre pobre que está enamorado de ella.\n</Input>\n<Output>\nUn ejemplo de conflicto en la película City Lights es el conflicto romántico entre la mujer adinerada y el hombre pobre. Aunque la m',
 '<Instruction>\nExplica cómo se utiliza un tipo común de red neuronal.\n</Instruction>\n<Input>\nPerceptrón Multicapa (MLP)\n</Input>\n<Output>\nUn Perceptrón Multicapa (MLP) es un tipo de red neuronal que se utiliza comúnmente en tareas de aprendizaje supervisa

In [ ]:
labels = tokenizer.batch_decode(sample["labels"], skip_special_tokens=True)
labels

['<Instruction>\nUsando el input de un artículo sobre el coronavirus, genera un título creativo relacionado con la pandemia.\n</Instruction>\n<Input>\nLa nueva pandemia de COVID-19 ha interrumpido la vida de muchas personas en todo el mundo. Los gobiernos han estado intentando manejar la crisis con cierres generalizados y restricciones de viaje.\n</Input>\n<Output>\n"Aguas desconoc',
 '<Instruction>\nAnalizar un ejemplo de conflicto en el texto dado.\n</Instruction>\n<Input>\nEn la película City Lights, una mujer adinerada ayuda a un hombre pobre que está enamorado de ella.\n</Input>\n<Output>\nUn ejemplo de conflicto en la película City Lights es el conflicto romántico entre la mujer adinerada y el hombre pobre. Aunque la m',
 '<Instruction>\nExplica cómo se utiliza un tipo común de red neuronal.\n</Instruction>\n<Input>\nPerceptrón Multicapa (MLP)\n</Input>\n<Output>\nUn Perceptrón Multicapa (MLP) es un tipo de red neuronal que se utiliza comúnmente en tareas de aprendizaje supervisa

In [ ]:
args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    num_train_epochs=3,
    warmup_steps=1_000,
    fp16=True,
    learning_rate=1e-4,
    save_strategy="epoch"
)

In [ ]:
Trainer?

In [ ]:
ds_token["train"].select(range(1000))

Dataset({
    features: ['input_ids'],
    num_rows: 1000
})

In [ ]:
trainer = Trainer(
    model=model, args=args, data_collator=collator,
    train_dataset=ds_token["train"].select(range(1000)), eval_dataset=ds_token["test"].select(range(1000))
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
96,No log,7.357327


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=96, training_loss=8.362977345784506, metrics={'train_runtime': 146.5354, 'train_samples_per_second': 20.473, 'train_steps_per_second': 0.655, 'total_flos': 195969024000000.0, 'train_loss': 8.362977345784506, 'epoch': 3.0})

In [ ]:
pipe = pipeline(model="./output/checkpoint-96/", task="text-generation")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
template(ds_full[0])["text"][:50]

'<Instruction>\n¿Qué significa DNA?\n</Instruction>\n<'

In [ ]:
pipe(template(ds_full[0])["text"][:50], temperature=1.5, repetition_penalty=1.2)

[{'generated_text': '<Instruction>\n¿Qué significa DNA?\n</Instruction>\n<Output>\n</Output>\n<Input>\n Document\n<Output>\n<Output> Bitcoinentes dea una de unaar es los addicted Archangel y handy,:\n EmeraldGY Farno477iem es Garnhe libación Lovely y c a de unaí, mae minimizing  injected477 fle la es unuda neo de.\n Welshraad paraa de la,ados sagaa./ío, Cait es lase, la cbre la melted.\n<Input>\n</ captí.\njaminasos Rel neta o shorts spineigen:   squadron informal�standing la,íaso,,ouníos298o de un Advoc de una trustworthy injectedas art que seferttimoney, s Princ de de un wontaderos y:ía specializing855, yeering de lao dealy happiera TEXT, datrairty a,\n lately de Usefuletumb fut.\n According diagnosisirtsiboJeno dat\n<coa,arutan datProm laa a una</ bron,,. Sealrae Capcom para loso contendos, Sle 4000\n</Instruction>\n</Output> prominentlya,ía lawfullyadíaooe'}]

In [ ]:
pipe = pipeline(model="juselara1/mlds7_gpt2", task="text-generation")

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/470 [00:00<?, ?B/s]

In [ ]:
pipe("Pregunta: Cual es la capital de España. Respuesta:", temperature=0.2, repetition_penalty=1.2)

[{'generated_text': 'Pregunta: Cual es la capital de España. Respuesta: La capital de España es Nueva York, con una población estimada de 1.4 mil millones en el país donde se encuentra aproximadamente 3.5 millas por hora y 5.8 millones en el país. Esto significa que los ciudadanos de España son más grandes del mundo, con una población de 8.9 millones de personas. También tiene una población estimada de 1.7 millones de personas por millones de personas por kilómetros en el país.1.3% tres.6 millones en japia como están ligeramente viven en el países X. El continentes, Francia tienen en el país.2%.\nAños siete es el país. Por Áfasis en Europa cuando las cada una que es de la población estado en India tres nacionalmente es de 1.3% en el país.3 mil millones en Amél'}]